In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(
    0,
    str(PROJECT_ROOT)
)

In [2]:
from src.persistence import (
    DEFAULT_DB_PATH,
    inicializar_banco,
    contar_transacoes,
    buscar_ultimas_transacoes
)

In [3]:
caminho_banco = (
    inicializar_banco()
)

print(
    "Banco criado em:",
    caminho_banco.resolve()
)

Banco criado em: /Users/lucassantos/Documents/ccard_fraud_ml/runtime/fraud_detection.db


In [4]:
print(
    "Banco existe:",
    DEFAULT_DB_PATH.exists()
)

print(
    "Transações armazenadas:",
    contar_transacoes()
)

Banco existe: True
Transações armazenadas: 0


In [5]:
buscar_ultimas_transacoes()

[]

In [6]:
from src.transaction_processor import (
    TransactionProcessor
)

In [7]:
processador = (
    TransactionProcessor()
)

In [8]:
import pandas as pd


DATA_PATH = (
    PROJECT_ROOT
    / "raw"
    / "fraudTrain.csv"
)


dtypes = {
    "cc_num": "string",
    "trans_num": "string",
    "zip": "string",
    "merchant": "string",
    "category": "category",
    "gender": "category",
    "state": "category",
    "first": "string",
    "last": "string",
    "street": "string",
    "city": "string",
    "job": "string"
}


df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    parse_dates=[
        "trans_date_trans_time",
        "dob"
    ]
)

In [9]:
colunas_indice = [
    coluna
    for coluna in df.columns
    if str(coluna).startswith("Unnamed:")
]

if colunas_indice:
    df = df.drop(
        columns=colunas_indice
    )

In [10]:
df = (
    df
    .sort_values(
        "trans_date_trans_time"
    )
    .reset_index(drop=True)
)

In [11]:
n_total = len(df)

fim_treino = int(
    n_total * 0.70
)

fim_validacao = int(
    n_total * 0.85
)

df_teste = (
    df.iloc[
        fim_validacao:
    ]
    .copy()
)

In [12]:
transacao_teste = (
    df_teste
    .iloc[[0]]
    .copy()
)

In [13]:
transacao_teste[
    [
        "trans_num",
        "trans_date_trans_time",
        "amt",
        "category",
        "is_fraud"
    ]
]

,trans_num,trans_date_trans_time,amt,category,is_fraud
1102173,b3cb6fd853393499393f26e9285d271f,2020-04-03 17:54:44,5.58,shopping_net,0


In [14]:
resultado = (
    processador
    .processar_transacao(
        transacao=transacao_teste,
        run_id="teste_individual_v1"
    )
)

In [15]:
from src.persistence import (
    contar_transacoes,
    buscar_ultimas_transacoes
)

In [16]:
print(
    "Registros da execução:",
    contar_transacoes(
        run_id="teste_individual_v1"
    )
)

Registros da execução: 1


In [17]:
buscar_ultimas_transacoes(
    limite=5,
    run_id="teste_individual_v1"
)

[{'id': 1,
  'run_id': 'teste_individual_v1',
  'trans_num': 'b3cb6fd853393499393f26e9285d271f',
  'trans_date_trans_time': '2020-04-03T17:54:44',
  'score_fraude': 2.055211055152726e-05,
  'decisao': 'APROVAR',
  'is_fraud': 0,
  'processed_at': '2026-08-17T00:55:31.050342+00:00',
  'latency_ms': 26.248458001646213,
  'model_version': 'catboost_v1',
  'policy_version': 'decision_policy_v1'}]

In [18]:
resultado_repetido = (
    processador
    .processar_transacao(
        transacao=transacao_teste,
        run_id="teste_individual_v1"
    )
)

In [19]:
resultado_repetido

{'run_id': 'teste_individual_v1',
 'trans_num': 'b3cb6fd853393499393f26e9285d271f',
 'trans_date_trans_time': Timestamp('2020-04-03 17:54:44'),
 'score_fraude': 2.055211055152726e-05,
 'decisao': 'APROVAR',
 'is_fraud': 0,
 'latency_ms': 5.229666989180259,
 'persistido': False}

In [20]:
contar_transacoes(
    run_id="teste_individual_v1"
)

1

### Processamento em lote

In [31]:
lote_teste = (
    df_teste
    .iloc[:1000]
    .copy()
)

In [32]:
lote_teste[
[
    "trans_num",
    "trans_date_trans_time",
    "amt",
    "category",
    "is_fraud"
]
]

,trans_num,trans_date_trans_time,amt,category,is_fraud
1102173,b3cb6fd853393499393f26e9285d271f,2020-04-03 17:54:44,5.58,shopping_net,0
1102174,590ef013c120f88c3147cf26ae7f9cfe,2020-04-03 17:54:44,771.85,misc_net,1
1102175,21c9bdef6b08b3628ffdf7e5d158264f,2020-04-03 17:55:17,202.82,kids_pets,0
1102176,d345b61a203a96885faa1d44a5bd815e,2020-04-03 17:55:35,2.33,shopping_pos,0
1102177,3d915c4e7c38d6ce14102421a4ce2d3c,2020-04-03 17:55:41,14.49,kids_pets,0
...,...,...,...,...,...
1103168,61bc8f8a9450ba5c510c21cab4dd6636,2020-04-04 07:00:26,73.63,grocery_pos,0
1103169,b6ab8000d9024032efba1a27fa332e70,2020-04-04 07:00:48,95.67,grocery_pos,0
1103170,4a079850bfee13c09dcf3e3cbdbcc898,2020-04-04 07:01:59,7.55,shopping_net,0
1103171,22bf0b4f096b8a16f281e9a645068889,2020-04-04 07:02:38,244.40,entertainment,0


In [33]:
RUN_ID = "teste_lote_10_v1"

In [34]:
resultados_lote = []

for i in range(len(lote_teste)):

    transacao = (
        lote_teste
        .iloc[[i]]
        .copy()
    )

    resultado = (
        processador
        .processar_transacao(
            transacao=transacao,
            run_id=RUN_ID
        )
    )

    resultados_lote.append(
        resultado
    )

In [41]:
df_resultados_1000 = pd.DataFrame(
    resultados_lote
)

In [42]:
print(
    "Persistidas:",
    contar_transacoes(
        run_id=RUN_ID
    )
)

print()

print(
    df_resultados_1000[
        "decisao"
    ].value_counts()
)

print()

print(
    df_resultados_1000[
        "latency_ms"
    ].describe()
)

Persistidas: 1000

decisao
APROVAR           992
ALERTA_CRITICO      7
REVISAR             1
Name: count, dtype: int64

count    1000.000000
mean        3.433138
std         1.000286
min         3.137042
25%         3.216302
50%         3.281208
75%         3.423542
max        32.499834
Name: latency_ms, dtype: float64


In [43]:
print(
    "Latência média:",
    f"{df_resultados_1000['latency_ms'].mean():.3f} ms"
)

print(
    "Latência mediana:",
    f"{df_resultados_1000['latency_ms'].median():.3f} ms"
)

print(
    "P95:",
    f"{df_resultados_1000['latency_ms'].quantile(0.95):.3f} ms"
)

print(
    "Máxima:",
    f"{df_resultados_1000['latency_ms'].max():.3f} ms"
)

Latência média: 3.433 ms
Latência mediana: 3.281 ms
P95: 4.089 ms
Máxima: 32.500 ms


In [35]:
df_resultados_lote = pd.DataFrame(
    resultados_lote
)

df_resultados_lote

,run_id,trans_num,trans_date_trans_time,score_fraude,decisao,is_fraud,latency_ms,persistido
0,teste_lote_10_v1,b3cb6fd853393499393f26e9285d271f,2020-04-03 17:54:44,0.000021,APROVAR,0,9.768708,False
1,teste_lote_10_v1,590ef013c120f88c3147cf26ae7f9cfe,2020-04-03 17:54:44,0.994915,ALERTA_CRITICO,1,3.837708,False
2,teste_lote_10_v1,21c9bdef6b08b3628ffdf7e5d158264f,2020-04-03 17:55:17,0.000086,APROVAR,0,4.100334,False
3,teste_lote_10_v1,d345b61a203a96885faa1d44a5bd815e,2020-04-03 17:55:35,0.000001,APROVAR,0,3.591625,False
4,teste_lote_10_v1,3d915c4e7c38d6ce14102421a4ce2d3c,2020-04-03 17:55:41,0.000299,APROVAR,0,3.964875,False
...,...,...,...,...,...,...,...,...
995,teste_lote_10_v1,61bc8f8a9450ba5c510c21cab4dd6636,2020-04-04 07:00:26,0.000008,APROVAR,0,3.182667,True
996,teste_lote_10_v1,b6ab8000d9024032efba1a27fa332e70,2020-04-04 07:00:48,0.000008,APROVAR,0,3.282875,True
997,teste_lote_10_v1,4a079850bfee13c09dcf3e3cbdbcc898,2020-04-04 07:01:59,0.000018,APROVAR,0,3.236542,True
998,teste_lote_10_v1,22bf0b4f096b8a16f281e9a645068889,2020-04-04 07:02:38,0.000076,APROVAR,0,3.313042,True


In [36]:
df_resultados_lote[
    [
        "trans_date_trans_time",
        "score_fraude",
        "decisao",
        "is_fraud",
        "latency_ms",
        "persistido"
    ]
]

,trans_date_trans_time,score_fraude,decisao,is_fraud,latency_ms,persistido
0,2020-04-03 17:54:44,0.000021,APROVAR,0,9.768708,False
1,2020-04-03 17:54:44,0.994915,ALERTA_CRITICO,1,3.837708,False
2,2020-04-03 17:55:17,0.000086,APROVAR,0,4.100334,False
3,2020-04-03 17:55:35,0.000001,APROVAR,0,3.591625,False
4,2020-04-03 17:55:41,0.000299,APROVAR,0,3.964875,False
...,...,...,...,...,...,...
995,2020-04-04 07:00:26,0.000008,APROVAR,0,3.182667,True
996,2020-04-04 07:00:48,0.000008,APROVAR,0,3.282875,True
997,2020-04-04 07:01:59,0.000018,APROVAR,0,3.236542,True
998,2020-04-04 07:02:38,0.000076,APROVAR,0,3.313042,True


In [37]:
contar_transacoes(
    run_id=RUN_ID
)

1000

In [38]:
buscar_ultimas_transacoes(
    limite=10,
    run_id=RUN_ID
)

[{'id': 1012,
  'run_id': 'teste_lote_10_v1',
  'trans_num': 'bd9702ad62c2c21d64e94f2e5127fc11',
  'trans_date_trans_time': '2020-04-04T07:02:51',
  'score_fraude': 1.7146206479760066e-05,
  'decisao': 'APROVAR',
  'is_fraud': 0,
  'processed_at': '2026-08-17T01:15:47.054853+00:00',
  'latency_ms': 3.391582999029197,
  'model_version': 'catboost_v1',
  'policy_version': 'decision_policy_v1'},
 {'id': 1011,
  'run_id': 'teste_lote_10_v1',
  'trans_num': '22bf0b4f096b8a16f281e9a645068889',
  'trans_date_trans_time': '2020-04-04T07:02:38',
  'score_fraude': 7.569917639551142e-05,
  'decisao': 'APROVAR',
  'is_fraud': 0,
  'processed_at': '2026-08-17T01:15:47.048397+00:00',
  'latency_ms': 3.3130420051747933,
  'model_version': 'catboost_v1',
  'policy_version': 'decision_policy_v1'},
 {'id': 1010,
  'run_id': 'teste_lote_10_v1',
  'trans_num': '4a079850bfee13c09dcf3e3cbdbcc898',
  'trans_date_trans_time': '2020-04-04T07:01:59',
  'score_fraude': 1.847772972031027e-05,
  'decisao': 'APROVA